# 08 — Conformal Prediction Intervals

**What:** Wrap the hybrid model's point forecast in distribution-free
prediction intervals using split conformal prediction. For any significance
level $\alpha \in (0,1)$, the interval $[L_{t+1}, U_{t+1}]$ satisfies:

$$P\left(y_{t+1} \in [L_{t+1}, U_{t+1}]\right) \geq 1 - \alpha$$

This guarantee holds in finite samples without any assumption on the
data-generating process.

**Why:** The hybrid model minimises RMSE but is slightly miscalibrated
on QLIKE — it underestimates tail risk. A risk manager acts on the upper
bound of a plausible range, not a point forecast. The upper bound
$U_{t+1}$ is the operationally relevant quantity for margin requirements
and hedge ratios. Standard GARCH intervals are asymptotic and assume
correct model specification — both are debatable here. Conformal
prediction requires neither.

**How:** All conformal logic is encapsulated in `src/models/conformal.py`.
The validation set residuals from notebook 07 serve as the calibration set.
This notebook calls the class interface and interprets findings.

**Outline:**

0. Setup
1. Theory
2. Calibration
3. Visualisation
4. Coverage Check
5. Export
6. Conclusion


---
---
## 0. Setup

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.models.conformal import ConformalPredictor

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 12, 'axes.labelsize': 11})

test_data = pd.read_csv(
    ROOT / 'data/processed/hybrid_predictions.csv',
    index_col='Date', parse_dates=True
)
val_data = pd.read_csv(
    ROOT / 'data/processed/hybrid_val_predictions.csv',
    index_col='Date', parse_dates=True
)

print(f'Calibration set : {len(val_data):,} rows | '
      f'{val_data.index[0].date()} → {val_data.index[-1].date()}')
print(f'Test set        : {len(test_data):,} rows | '
      f'{test_data.index[0].date()} → {test_data.index[-1].date()}')


---
---
## 1. Theory

**Split conformal prediction** computes nonconformity scores on a
held-out calibration set and uses their empirical quantile as the
interval half-width. For regression, the nonconformity score is:

$$s_i = |y_i - \hat{y}_i|, \quad i \in \mathcal{D}_{\text{cal}}$$

The prediction interval for a new point is:

$$C_\alpha = [\hat{y}_{T+1} - \hat{q},\; \hat{y}_{T+1} + \hat{q}]$$

where $\hat{q} = \text{Quantile}(\{s_i\}, 1-\alpha)$. The interval
width is constant across all test points — the method produces marginal,
not conditional, coverage.

**Why the validation set is the calibration set:** the validation set
was used only for XGBoost early stopping. The model never fitted to it,
so its residuals are exchangeable with the test residuals under
stationarity — the standard split conformal assumption.


---
---
## 2. Calibration

In [ ]:
cp = ConformalPredictor()
cp.calibrate(
    y_true = val_data['y_true'].values,
    y_pred = val_data['y_pred_hybrid'].values,
)

print(cp)
print()
for alpha in [0.20, 0.10, 0.05]:
    q = cp.quantile(alpha)
    print(f'alpha={alpha:.2f} | coverage={1-alpha:.0%} | '
          f'quantile={q:.6f} | width={2*q:.6f}')


---
---
## 3. Visualisation

The hybrid point forecast with three nested conformal intervals at
80\%, 90\%, and 95\% coverage. Actual realised values (crimson)
should fall outside the 80\% band ~20\% of the time, outside the
90\% band ~10\% of the time, and outside the 95\% band ~5\% of the time.


In [ ]:
y_pred_test = test_data['y_pred_hybrid'].values
y_true_test = test_data['y_true'].values
test_index  = test_data.index

results = cp.predict_all(y_pred_test)

fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(test_index,
                results['0.05']['lower'], results['0.05']['upper'],
                alpha=0.15, color='steelblue', label='95% interval')
ax.fill_between(test_index,
                results['0.1']['lower'],  results['0.1']['upper'],
                alpha=0.25, color='steelblue', label='90% interval')
ax.fill_between(test_index,
                results['0.2']['lower'],  results['0.2']['upper'],
                alpha=0.40, color='steelblue', label='80% interval')

ax.plot(test_index, y_pred_test, color='steelblue',
        linewidth=0.8, label='Hybrid forecast')
ax.plot(test_index, y_true_test, color='crimson',
        linewidth=0.6, alpha=0.8, label='Actual |r_{t+1}|')

ax.set_title('Hybrid Volatility Forecast with Conformal Prediction Intervals')
ax.set_ylabel('Daily Absolute Return')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()


---
---
## 4. Coverage Check

Empirical coverage must be at least the nominal level for a valid
conformal predictor. A negative gap means the intervals are conservative.
A positive gap would indicate undercoverage — a violation of the guarantee.


In [ ]:
display(cp.coverage_summary(y_true_test, y_pred_test))


---
---
## 5. Export

In [ ]:
conformal_intervals = pd.DataFrame({
    'y_true'   : y_true_test,
    'y_pred'   : y_pred_test,
    'lower_80' : results['0.2']['lower'],
    'upper_80' : results['0.2']['upper'],
    'lower_90' : results['0.1']['lower'],
    'upper_90' : results['0.1']['upper'],
    'lower_95' : results['0.05']['lower'],
    'upper_95' : results['0.05']['upper'],
}, index=test_index)

conformal_intervals.to_csv(ROOT / 'data/processed/conformal_intervals.csv')
print(f'Intervals saved → {len(conformal_intervals):,} rows | '
      f'{conformal_intervals.index[0].date()} → '
      f'{conformal_intervals.index[-1].date()}')

cp.save(ROOT / 'data/processed/conformal.pkl')
display(conformal_intervals.head())


---
---
## 6. Conclusion

The split conformal predictor is calibrated on 987 validation observations
(2018–2022) and evaluated on 987 test observations (2022–2026).

**Coverage results.** All three nominal coverage levels are met or
exceeded. At 80\%, empirical coverage is 80.67\% (gap −0.67\%).
At 90\%, empirical coverage is 92.81\% (gap −2.81\%) — the intervals
are slightly conservative at this level. At 95\%, empirical coverage
is 95.95\% (gap −0.95\%). The conformal guarantee is satisfied at
all three levels, confirming that the validation residuals are
exchangeable with the test residuals as required.

**Interval widths.** The 90\% interval half-width is 0.0068 in daily
absolute return units — meaning the hybrid forecast is typically within
$\pm 0.68$ percentage points of the actual next-day move at 90\%
confidence. For a currency pair with annualised volatility around 10\%,
this is a tight and operationally useful interval.

**Constant width.** Because split conformal produces a fixed scalar
quantile, every test day gets the same interval width regardless of
the current volatility regime. This is a property of marginal coverage.
An adaptive interval — wider during crises, narrower during calm periods
— would require locally weighted conformal prediction, documented as a
natural extension in the Future Perspectives section.

**Operational interpretation.** The upper bound $U_{t+1}$ of the 90\%
interval is the number a risk manager would use to set margin requirements.
A position sized assuming no more than the 90\% upper bound in daily
moves will be correctly hedged on 9 out of 10 trading days — with no
distributional assumption required. This is the output that converts
the hybrid forecast into a risk management tool.

**What comes next.** Notebook 09 formally backtests these intervals
using the Kupiec Proportion of Failures (POF) test and converts the
GARCH conditional volatility into a 99\% Value-at-Risk estimate for
Basel traffic light backtesting.
